# 과제 2: SageMaker 처리 및 훈련 작업 사용
이 과제에서는 데이터 처리, 피처 엔지니어링 및 모델 훈련 코드를 SageMaker 작업으로 이동합니다.

다음 다이어그램은 SageMaker 컨테이너의 구조를 보여줍니다:

![](../img/container-anatomy.png)

이 과제의 코드 스니펫과 일반적인 지침은 [`02-sagemaker-containers.ipynb`](../02-sagemaker-containers.ipynb) 노트북을 참조하세요.

## 패키지 임포트

In [ ]:
import time
import boto3
import botocore
import numpy as np  
import pandas as pd  
import sagemaker
from time import gmtime, strftime, sleep
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sklearn.metrics import roc_auc_score
from smexperiments.experiment import Experiment
from smexperiments.trial import Trial
from smexperiments.trial_component import TrialComponent
from smexperiments.tracker import Tracker

sagemaker.__version__

In [ ]:
session = sagemaker.Session()
sm = session.sagemaker_client

## [선택사항] 기존 실험 로드 또는 새 실험 생성
이 노트북에서 매개변수, 메트릭 및 아티팩트를 추적하기 위해 기존 실험을 로드하거나 새 실험을 생성합니다.

In [ ]:
# 이름을 기반으로 실험 로드
# experiment = Experiment.load(experiment_name, sagemaker_boto_client=sm)

In [ ]:
# 또는 새 실험 생성
#experiment_name = f"from-idea-to-prod-experiment-{strftime('%d-%H-%M-%S', gmtime())}"
#experiment = Experiment.create(
#    experiment_name=experiment_name,
#    description="Direct marketing binary classification",
#    sagemaker_boto_client=sm,
#)

## 연습 1: 데이터 처리
- SageMaker 세션 객체를 사용하여 데이터셋을 Amazon S3 버킷에 [업로드](https://sagemaker.readthedocs.io/en/stable/api/utility/session.html#sagemaker.session.Session.upload_data)합니다. SageMaker [기본 버킷](https://sagemaker.readthedocs.io/en/stable/api/utility/session.html#sagemaker.session.Session.default_bucket) 사용
- 이전 노트북의 데이터 처리 코드를 Python 실행 가능 스크립트로 이동합니다. 스크립트에 매개변수를 전달하여 데이터 처리를 매개변수화할 수 있습니다
- 출력 데이터셋의 Amazon S3 경로 설정
- [SageMaker Python SDK](https://sagemaker.readthedocs.io/en/stable/overview.html) [`SKLearnProcessor`](https://sagemaker.readthedocs.io/en/stable/frameworks/sklearn/sagemaker.sklearn.html#sagemaker.sklearn.processing.SKLearnProcessor) 클래스를 사용하여 처리 작업 설정
- 처리 작업의 [입력](https://sagemaker.readthedocs.io/en/stable/api/training/processing.html#sagemaker.processing.ProcessingInput) 및 [출력](https://sagemaker.readthedocs.io/en/stable/api/training/processing.html#sagemaker.processing.ProcessingOutput)을 구성하여 Amazon S3 위치를 가리키도록 설정
- 처리 작업 [실행](https://sagemaker.readthedocs.io/en/stable/api/training/processing.html#sagemaker.processing.ScriptProcessor.run)

### Python SDK 프로세서 클래스
사용 사례에 가장 적합한 클래스를 사용하여 프로세서를 구현합니다:
    
![](../img/python-sdk-processors.png)

In [ ]:
session = sagemaker.Session()

In [ ]:
# 데이터 업로드 코드 작성
# 전체 데이터셋에 대한 S3 키
# input_s3_url = session.upload_data()


In [ ]:
%%writefile preprocessing_assignment.py

# 실행 가능한 데이터 처리 코드를 여기에 작성
import pandas as pd
import numpy as np
import argparse
import os

def _parse_args():
    
    parser = argparse.ArgumentParser()
    # 데이터, 모델 및 출력 디렉토리
    # model_dir은 항상 SageMaker에서 전달됩니다. 기본적으로 이것은 기본 버킷 아래의 S3 경로입니다.
    parser.add_argument('--filepath', type=str, default='/opt/ml/processing/input/')
    parser.add_argument('--filename', type=str, default='bank-additional-full.csv')
    parser.add_argument('--outputpath', type=str, default='/opt/ml/processing/output/')
    
    return parser.parse_known_args()


if __name__=="__main__":
    # 인수 처리
    args, _ = _parse_args()
    
    print("데이터 처리 및 피처 엔지니어링 시작")
    
    # 로컬 파일에서 데이터 로드
    
    # 처리 및 피처 엔지니어링 코드 여기에
    
    # 데이터 분할
    
    # 데이터셋(train, validation, test, baseline)을 로컬에 저장
    
    print("## 처리 완료. 종료.")

In [ ]:
# 출력 데이터셋의 Amazon S3 경로 설정
# train_s3_url = 
# validation_s3_url = 
# test_s3_url = 
# baseline_s3_url = 


### [선택사항] 시행(Trial) 생성
실험을 사용하는 경우, 이 노트북의 처리 및 훈련 출력을 캡처하기 위해 시행(trial)을 생성해야 합니다.

[`Trial`](https://sagemaker-experiments.readthedocs.io/en/latest/trial.html) 클래스를 사용하여 시행과 상호 작용하고 [`Tracker`](https://sagemaker-experiments.readthedocs.io/en/latest/tracker.html) 클래스를 사용하여 시행 구성요소에 정보를 기록합니다.

SageMaker 처리 및 훈련 작업은 `Processor.run()` 또는 `Estimator.fit()` 호출에서 `experiment_config`를 제공하면 시행 구성요소를 자동으로 처리하고 메트릭, 매개변수, 메타데이터 및 아티팩트를 시행 구성요소에 저장합니다.

In [ ]:
# trial = experiment.create_trial(trial_name_prefix="Container-training")

In [ ]:
# with Tracker.create(display_name="Preprocessing-split", sagemaker_boto_client=sm) as tracker:
#    tracker.log_parameters()
#    tracker.log_input()

In [ ]:
# 처리 및 훈련 작업에 사용할 실험 구성 생성
#experiment_config = {
#    "ExperimentName": experiment.experiment_name,
#    "TrialName": trial.trial_name,
#    "TrialComponentDisplayName": "Preprocessing",
#}

### 프로세서 생성

In [ ]:
# SKLearnProcessor 생성
framework_version = "0.23-1"
processing_instance_type = "ml.m5.large"
processing_instance_count = 1

# sklearn_processor = SKLearnProcessor()


In [ ]:
# 처리 입력 및 출력 정의
processing_inputs = [] # 전체 데이터셋에 대한 포인터로 input_s3_url 사용

processing_outputs = [] # 처리 컨테이너의 로컬 디렉토리를 Amazon S3 위치에 매핑

In [ ]:
# 처리 작업 시작, 실험을 사용하는 경우 experiment_config 매개변수 전달
# sklearn_processor.run() 

## 연습 2: 모델 훈련
- SageMaker SDK [헬퍼](https://sagemaker.readthedocs.io/en/stable/api/utility/image_uris.html#sagemaker.image_uris.retrieve)를 사용하여 사용된 내장 SageMaker ML 알고리즘의 컨테이너 이미지 URI 가져오기
- 훈련 작업에 대한 데이터 [입력 채널](https://sagemaker.readthedocs.io/en/stable/api/utility/inputs.html#sagemaker.inputs.TrainingInput) 구성
- [`Estimator`](https://sagemaker.readthedocs.io/en/stable/api/training/estimators.html#sagemaker.estimator.Estimator) 클래스를 사용하여 훈련 작업 설정
- [하이퍼파라미터](https://sagemaker.readthedocs.io/en/stable/api/training/estimators.html#sagemaker.estimator.Estimator.set_hyperparameters) 설정
- 훈련 작업 [실행](https://sagemaker.readthedocs.io/en/stable/api/training/estimators.html#sagemaker.estimator.EstimatorBase.fit)

In [ ]:
# 컨테이너 이미지 URI를 검색하는 코드 작성


In [ ]:
# 입력 데이터 채널 설정
# s3_input_train = 
# s3_input_validation =

In [ ]:
# 모델 아티팩트의 Amazon S3 경로 설정
# output_s3_url = 

### Python SDK 추정기(Estimator) 클래스
SageMaker Python SDK에는 각 내장 알고리즘에 액세스하기 위한 해당 [`EstimatorBase`](https://sagemaker.readthedocs.io/en/stable/api/training/estimators.html#sagemaker.estimator.EstimatorBase) 파생 클래스가 포함되어 있습니다. [`Framework`](https://sagemaker.readthedocs.io/en/stable/api/training/estimators.html#sagemaker.estimator.Framework) 클래스를 확장하여 사용자 정의 프레임워크로 훈련을 구현할 수 있습니다.

![](../img/python-sdk-estimators.png)

In [ ]:
# 추정기 생성
train_instance_count = 1
train_instance_type = "ml.m5.xlarge"

# estimator = sagemaker.estimator.Estimator()

In [ ]:
# 추정기 알고리즘의 하이퍼파라미터 설정
# estimator.set_hyperparameters()

In [ ]:
# 훈련 입력 설정
# training_inputs = {}

In [ ]:
# 훈련 작업 실행, 선택적으로 experiment_config 매개변수 사용
# estimator.fit(training_inputs)

훈련 작업이 완료될 때까지 기다립니다.

In [ ]:
# 훈련 작업 설명
# training_job_name = estimator._current_job_name
# boto3.client("sagemaker", region_name=region).describe_training_job(TrainingJobName=training_job_name)

In [ ]:
# describe job 결과에서 모델 메트릭 가져오기

# print(f"Train-auc:{train_auc:.2f}, Validate-auc:{validate_auc:.2f}")

## 연습 3: 모델 검증
모델을 검증하려면 훈련 작업의 모델 아티팩트를 사용하여 테스트 데이터셋에서 예측을 실행합니다. [실시간 추론 엔드포인트](https://docs.aws.amazon.com/sagemaker/latest/dg/realtime-endpoints.html)를 생성하거나 [배치 변환](https://docs.aws.amazon.com/sagemaker/latest/dg/batch-transform.html)을 생성할 수 있습니다.

### 옵션 1: 실시간 추론
- [Estimator.deploy](https://sagemaker.readthedocs.io/en/stable/api/training/estimators.html#sagemaker.estimator.EstimatorBase.deploy) 함수를 사용하여 실시간 추론 엔드포인트 프로비저닝
- 테스트 데이터셋 로드
- 테스트 데이터셋을 엔드포인트로 전송. [Predictor.predict](https://sagemaker.readthedocs.io/en/stable/api/inference/predictors.html#sagemaker.predictor.Predictor.predict) 함수 사용
- 예측 평가

In [ ]:
# 예측기 생성
# 참고: 훈련 작업은 지정된 S3 위치에 테스트 데이터셋을 저장했습니다

# predictor = estimator.deploy()

In [ ]:
# 테스트 데이터셋 로드
# test_x = pd.read_csv()
# test_y = pd.read_csv()

In [ ]:
# 예측


In [ ]:
# 예측 평가
# 예측된 레이블을 실제 레이블과 비교


### 옵션 2: 배치 변환
비동기 추론의 경우 SageMaker [변환 작업](https://docs.aws.amazon.com/sagemaker/latest/dg/batch-transform.html)을 사용할 수 있습니다.
- [Estimator.transformer](https://sagemaker.readthedocs.io/en/stable/api/training/estimators.html#sagemaker.estimator.EstimatorBase.transformer) 함수를 사용하여 변환기 생성
- 변환 작업 [실행](https://sagemaker.readthedocs.io/en/stable/api/inference/transformer.html#sagemaker.transformer.Transformer.transform)
- S3 출력 위치에서 데이터셋 다운로드
- 예측 평가

In [ ]:
# 변환기 생성
# transformer = estimator.transformer()

In [ ]:
# 변환 실행, experiment_config 매개변수 사용
# transformer.transform()

변환 작업이 완료될 때까지 기다립니다.

변환기는 예측 확률을 출력하고 지정된 S3 위치에 `csv` 파일로 저장합니다. S3 경로는 `transformer.output_path`에 저장됩니다. 예측을 실제 레이블과 비교하려면 S3에서 데이터셋을 다운로드하고 Pandas DataFrame에 로드해야 합니다.

In [ ]:
# S3에서 예측 및 실제 레이블 다운로드


In [ ]:
# 출력 데이터셋 및 실제 레이블 로드
# predictions = pd.read_csv()
# test_y = pd.read_csv()

In [ ]:
# 혼동 행렬 표시
# pd.crosstab()


In [ ]:
# AUC 계산
# test_auc = roc_auc_score(test_y, predictions)
#  print(f"Test-auc: {test_auc:.2f}")

### [선택사항] ROC 및 정밀도-재현율 곡선 작성
[`sklearn.metrics`](https://scikit-learn.org/stable/modules/model_evaluation.html) 패키지를 사용하여 다양한 차트를 생성할 수 있습니다.

### [선택사항] 시행 구성요소에 차트 저장
Tracker 클래스를 사용하여 실험의 시행(trial)의 시행 구성요소(trial component)에 다양한 차트를 저장할 수 있습니다.

Jupyter 노트북 팁: `Ctrl` + `/`를 눌러 셀에서 선택한 모든 줄을 주석 처리하거나 주석 해제합니다.

In [ ]:
# 표시 이름을 기반으로 현재 시행의 시행 구성요소 이름 찾기
#batch_transform_trail_component = [
#    tc for tc in trial.list_trial_components() 
#    if tc.display_name == <DISPLAY NAME OF THE TRIAL COMPONENT>][0]

In [ ]:
# 차트 추가
# with Tracker.load(
#    trial_component_name=batch_transform_trail_component.trial_component_name,
#    sagemaker_boto_client=sm
# ) as tracker:
#    tracker.log_precision_recall()
#    tracker.log_confusion_matrix()
#    tracker.log_roc_curve()

### [선택사항] Studio에서 실험, 시행 및 시행 구성요소 탐색
**SageMaker 리소스**에서 **실험 및 시행**을 선택하고, 컨텍스트 메뉴에서 **시행 구성요소 목록에서 열기**를 선택합니다:

<img src="../img/experiment-and-trials-context-menu.png" width="400"/>

## 연습 4: [선택사항] 하이퍼파라미터 최적화 (HPO)
- [HyperparameterTuner](https://sagemaker.readthedocs.io/en/stable/api/training/tuner.html#sagemaker.tuner.HyperparameterTuner)를 사용하여 HPO 작업 실행
- 하이퍼파라미터 범위 및 튜닝 전략 지정
- 튜닝 작업 [실행](https://sagemaker.readthedocs.io/en/stable/api/training/tuner.html#sagemaker.tuner.HyperparameterTuner.fit)
- 튜닝된 모델과 튜닝되지 않은 모델의 성능 비교

In [ ]:
# 필요한 HPO 객체 임포트
from sagemaker.tuner import (
    CategoricalParameter,
    ContinuousParameter,
    HyperparameterTuner,
    IntegerParameter,
)

In [ ]:
# 하이퍼파라미터 범위 설정
# hp_ranges = {}


In [ ]:
# 목적 메트릭 설정
objective = "validation:auc"

In [ ]:
# HPO 객체 인스턴스화
# tuner = HyperparameterTuner()

In [ ]:
# 성능 평가

## 정리
생성한 모든 실시간 엔드포인트 제거

In [ ]:
# predictor.delete_endpoint(delete_endpoint_config=True)


In [ ]:
# HPO 후 튜닝된 예측기를 생성한 경우 실행
# hpo_predictor.delete_endpoint(delete_endpoint_config=True)


## 과제 3 계속하기
[과제 3](03-assignment-sagemaker-pipeline.ipynb) 노트북으로 이동하세요.